# CS03c — Signed futures flow vs the perpetual's own next 5-minute return

Protocol: `discussions/Candle_Streaks/003c_claude_perp_target_preregistration.md`.

CS03b ablation, frozen, with the target market switched from Binance spot to the Binance USD-M perpetual (sign, streak, volatility and target from perp closes). Spot target is refit on the **same days** as a lead-lag reference. Economic test: hold $\operatorname{sign}(P_{up}-P_{down})$ for 5 minutes when $|P_{up}-P_{down}|$ is in the TRAIN top decile, net of per-side costs.

In [1]:
import os,sys,json,subprocess,importlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Execute notebook inside the repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import candle_streak_trade_flow as tf
from data_engineering.modeling import candle_streak_flow_attribution as ab
from data_engineering.modeling import candle_streak_perp_target as pt
for m in (tf,ab,pt):importlib.reload(m)
assert pt.VERSION=='CS03c-0.1.0', pt.VERSION
cfg=pt.Specification();abcfg=ab.Specification()
SYNTHETIC=os.getenv('CS03C_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='cs03c-');OUT=Path(tmp.name)/'03c'
else:
    from data_engineering.modeling import markov_lift_distributions as vd14
    OUT=ROOT/'notebooks'/'Candle_Streaks'/'03c'
OUT.mkdir(parents=True,exist_ok=True)
print('CS03c:',pt.VERSION,'CS03b:',ab.VERSION,'CS03:',tf.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
checks=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_perp_target.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(checks.stdout[-2000:]);print(checks.stderr[-1000:])
assert checks.returncode==0, 'CS03c tests failed; stop before historical evaluation'

CS03c: CS03c-0.1.0 CS03b: CS03b-0.1.0 CS03: CS03-0.1.1 synthetic: False output: D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03c
........                                                                 [100%]
8 passed in 5.84s




In [2]:
ASSET_DATA={};excluded={}
if SYNTHETIC:
    kl,sp=pt.synthetic_pair(n_days=130,seed=2606,perp_beta=0.0,spot_lag_beta=0.6)   # pure lead-lag: spot-only signal
    pr,ff=pt.perp_returns_5m(kl),tf.one_minute_futures_flow(kl)
    ASSET_DATA={'BTC':(pr,ff,sp)}
    data_description='SYNTHETIC SMOKE DATA (pure perp->spot lead-lag) — NOT HISTORICAL'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT not set in .env/environment')
    root=Path(rootval);hist=root/'history'/'price_5m'
    futures=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else root.parent/'Binance'
    print('Perp input:',futures,'spot reference input:',hist)
    for asset in cfg.assets:
        path=futures/f'{asset}USDT_um_1m.parquet'
        if not path.is_file():excluded[asset]=f'Missing {path}';continue
        try:
            pr,ff=pt.read_perp(path,end=cfg.end)
            try:
                prices,_=vd14.load_returns(hist,(asset,),end=pd.Timestamp(cfg.end),min_days=365)
                sp=prices.loc[prices.underlying.eq(asset),['state_time','return_5m']].copy()
            except Exception as exc:
                print(asset,'spot reference unavailable:',exc);sp=None
            ASSET_DATA[asset]=(pr,ff,sp)
        except Exception as exc:
            excluded[asset]=f'{type(exc).__name__}: {exc}'
    data_description='LOCAL Binance USD-M 1m (perp target and flow); Binance spot 5m reference'
assert ASSET_DATA,f'No input coverage: {excluded}'
print('Available:',list(ASSET_DATA),'excluded:',excluded)

Perp input: D:\MarketData\Binance spot reference input: D:\MarketData\Deribit\history\price_5m
Available: ['BTC', 'ETH', 'SOL', 'XRP'] excluded: {}


In [3]:
prepared={};splits={};quality=[]
for asset,(pr,ff,sp) in ASSET_DATA.items():
    panels={'perp':pt.build_panel(pr,ff)}
    if sp is not None:panels['spot']=pt.build_panel(sp,ff)
    dates=None
    for x in panels.values():
        d=set(tf.full_days(x));dates=d if dates is None else dates&d
    dates=np.array(sorted(dates))
    quality.append({'asset':asset,'perp_5m_rows':len(pr),'perp_valid_returns':int(np.isfinite(pr.return_5m).sum()),
                    'flow_valid':int(ff.flow_valid.sum()),'targets':list(panels),'common_complete_days':len(dates),
                    'first':str(dates.min()) if len(dates) else None,'last':str(dates.max()) if len(dates) else None})
    if len(dates)<60:excluded[asset]=f'Only {len(dates)} common complete days';continue
    for tgt,x in panels.items():
        data,split=tf.split_panel(x,dates,abcfg.train_fraction,abcfg.valid_fraction)
        data['asset']=asset;prepared[(asset,tgt)]=data;splits[f'{asset}_{tgt}']=split
display(pd.DataFrame(quality))
print('Panels:',list(prepared))
if not SYNTHETIC and not all((a,'perp') in prepared for a in cfg.primary_assets):print('PRIMARY BTC/ETH PERP COVERAGE MISSING')

,asset,perp_5m_rows,perp_valid_returns,flow_valid,targets,common_complete_days,first,last
0,BTC,213408,213407,213387,"[perp, spot]",735,2024-10-03 00:00:00+00:00,2026-10-10 00:00:00+00:00
1,ETH,213408,213407,213387,"[perp, spot]",735,2024-10-03 00:00:00+00:00,2026-10-10 00:00:00+00:00
2,SOL,213408,213407,213387,"[perp, spot]",735,2024-10-03 00:00:00+00:00,2026-10-10 00:00:00+00:00
3,XRP,213408,213407,213387,"[perp, spot]",735,2024-10-03 00:00:00+00:00,2026-10-10 00:00:00+00:00


Panels: [('BTC', 'perp'), ('BTC', 'spot'), ('ETH', 'perp'), ('ETH', 'spot'), ('SOL', 'perp'), ('SOL', 'spot'), ('XRP', 'perp'), ('XRP', 'spot')]


In [4]:
B=cfg.bootstrap_reps if not SYNTHETIC else 100
all_scores=[];all_boot=[];all_econ=[];all_daily=[];thresholds=[]
for (asset,tgt),x in prepared.items():
    train=x[x.period.eq('train')]
    fit=ab.fit_ablation(train,C=abcfg.logistic_C,max_iter=abcfg.max_iter,tol=abcfg.tol)
    thr={m:pt.edge_threshold(train,fit,m,cfg.edge_quantile) for m in pt.ECON_MODELS}
    thresholds+= [{'asset':asset,'target':tgt,'model':m,'q90_abs_edge_train':v} for m,v in thr.items()]
    for period in ('valid','test'):
        ev=x[x.period.eq(period)&x.scorable].copy()
        p=ab.predict_ablation(ev,fit)
        losses=ab.origin_losses(ev,p)
        all_scores.append(ab.scores_from_losses(losses).assign(asset=asset,target=tgt,period=period))
        all_boot.append(ab.day_bootstrap(losses,B=B,seed=abcfg.seed,min_days=abcfg.min_days_ci,
                                         min_events=abcfg.min_events_ci).assign(asset=asset,target=tgt,period=period))
        for m in pt.ECON_MODELS:
            e=pt.edge(p[m])
            for rule,t in (('top_decile',thr[m]),('every_bar',0.0)):
                for cname,c in pt.cost_scenarios(cfg).items():
                    d=pt.daily_pnl(ev,e,t,c)
                    all_econ.append({'asset':asset,'target':tgt,'period':period,'model':m,'rule':rule,'cost':cname,
                                     'cost_per_side_bp':1e4*c,**pt.econ_summary(d,B,cfg.seed)})
                    if rule=='top_decile' and cname=='taker' and period=='test':
                        all_daily.append(d.assign(asset=asset,target=tgt,model=m))
        del losses,p,ev
scores=pd.concat(all_scores,ignore_index=True);bootstrap=pd.concat(all_boot,ignore_index=True)
econ=pd.DataFrame(all_econ);daily=pd.concat(all_daily,ignore_index=True);thresholds=pd.DataFrame(thresholds)
display(thresholds)

,asset,target,model,q90_abs_edge_train
0,BTC,perp,price_vol,0.051972
1,BTC,perp,price_vol_signed,0.068076
2,BTC,perp,price_vol_both,0.069886
3,BTC,spot,price_vol,0.045048
4,BTC,spot,price_vol_signed,0.062310
5,BTC,spot,price_vol_both,0.063733
6,ETH,perp,price_vol,0.078883
7,ETH,perp,price_vol_signed,0.083712
8,ETH,perp,price_vol_both,0.085448
9,ETH,spot,price_vol,0.071901


## Statistical: signed flow vs price-only, TEST (negative Δ = signed flow helps)

In [5]:
tb=bootstrap[(bootstrap.period=='test')&bootstrap.metric.isin(['threeclass_logloss','binary_logloss'])&
             (bootstrap.reference=='price_vol')&bootstrap.model.isin(['price_vol_signed','price_vol_both'])]
display(tb[['asset','target','model','metric','n_origins','delta_equal_day','ci025','ci975','status']].sort_values(['metric','model','asset','target']))

,asset,target,model,metric,n_origins,delta_equal_day,ci025,ci975,status
74,BTC,perp,price_vol_both,binary_logloss,42176,-0.000310,-0.000584,-0.000034,OK
194,BTC,spot,price_vol_both,binary_logloss,42144,-0.000244,-0.000510,0.000005,OK
314,ETH,perp,price_vol_both,binary_logloss,42224,-0.000586,-0.000892,-0.000267,OK
434,ETH,spot,price_vol_both,binary_logloss,42193,-0.000610,-0.000940,-0.000292,OK
554,SOL,perp,price_vol_both,binary_logloss,40592,0.000170,-0.000061,0.000385,OK
674,SOL,spot,price_vol_both,binary_logloss,40534,0.000141,-0.000107,0.000385,OK
794,XRP,perp,price_vol_both,binary_logloss,40936,-0.000106,-0.000327,0.000106,OK
914,XRP,spot,price_vol_both,binary_logloss,40924,-0.000130,-0.000361,0.000103,OK
72,BTC,perp,price_vol_signed,binary_logloss,42176,-0.000365,-0.000597,-0.000135,OK
192,BTC,spot,price_vol_signed,binary_logloss,42144,-0.000275,-0.000493,-0.000073,OK


## Economic: TEST, top-decile rule (bp; per-day CI from UTC-day bootstrap)

In [6]:
cols=['asset','target','model','cost','active_bars','hit_rate','gross_bp_per_active_bar','turnover_per_day',
      'gross_bp_per_day','cost_bp_per_day','net_bp_per_day','net_bp_per_day_lo','net_bp_per_day_hi']
display(econ[(econ.period=='test')&(econ.rule=='top_decile')][cols].sort_values(['cost','model','asset','target']).round(3))
print('Every-bar rule, zero cost (pure gross edge):')
display(econ[(econ.period=='test')&(econ.rule=='every_bar')&(econ.cost=='zero')][cols].round(3))

,asset,target,model,cost,active_bars,hit_rate,gross_bp_per_active_bar,turnover_per_day,gross_bp_per_day,cost_bp_per_day,net_bp_per_day,net_bp_per_day_lo,net_bp_per_day_hi
19,BTC,perp,price_vol,low_fee,4185,0.513,0.187,30.367,5.334,91.102,-85.768,-97.316,-74.371
55,BTC,spot,price_vol,low_fee,4310,0.514,0.263,31.429,7.700,94.286,-86.586,-99.838,-72.655
91,ETH,perp,price_vol,low_fee,4212,0.536,0.116,31.592,3.320,94.776,-91.456,-107.819,-76.913
127,ETH,spot,price_vol,low_fee,4209,0.532,-0.069,30.721,-1.973,92.163,-94.137,-112.190,-79.495
163,SOL,perp,price_vol,low_fee,3876,0.519,0.518,29.279,13.665,87.837,-74.172,-91.124,-58.925
...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,ETH,spot,price_vol_signed,zero,4783,0.547,0.197,36.259,6.404,0.000,6.404,-8.638,21.066
170,SOL,perp,price_vol_signed,zero,4752,0.508,0.383,37.333,12.365,0.000,12.365,-3.795,27.874
206,SOL,spot,price_vol_signed,zero,4984,0.507,0.343,40.680,11.622,0.000,11.622,-8.191,29.804
242,XRP,perp,price_vol_signed,zero,4354,0.506,0.164,32.789,4.848,0.000,4.848,-11.739,19.291


Every-bar rule, zero cost (pure gross edge):


,asset,target,model,cost,active_bars,hit_rate,gross_bp_per_active_bar,turnover_per_day,gross_bp_per_day,cost_bp_per_day,net_bp_per_day,net_bp_per_day_lo,net_bp_per_day_hi
23,BTC,perp,price_vol,zero,42335,0.504,0.071,295.932,20.410,0.0,20.410,-15.711,54.483
29,BTC,perp,price_vol_signed,zero,42335,0.509,0.129,199.020,37.057,0.0,37.057,4.322,69.367
35,BTC,perp,price_vol_both,zero,42335,0.510,0.095,194.272,27.281,0.0,27.281,-3.574,58.643
59,BTC,spot,price_vol,zero,42335,0.501,0.071,296.694,20.405,0.0,20.405,-11.727,49.471
65,BTC,spot,price_vol_signed,zero,42335,0.507,0.131,217.184,37.681,0.0,37.681,6.220,69.307
71,BTC,spot,price_vol_both,zero,42335,0.506,0.090,211.524,26.049,0.0,26.049,-4.177,56.735
95,ETH,perp,price_vol,zero,42335,0.501,-0.042,297.959,-12.089,0.0,-12.089,-58.223,31.301
101,ETH,perp,price_vol_signed,zero,42335,0.513,-0.014,177.170,-4.082,0.0,-4.082,-50.697,38.376
107,ETH,perp,price_vol_both,zero,42335,0.511,-0.040,175.986,-11.376,0.0,-11.376,-55.054,28.277
131,ETH,spot,price_vol,zero,42335,0.501,-0.023,293.469,-6.658,0.0,-6.658,-53.470,34.757


In [7]:
dec=pt.decision(bootstrap,econ,cfg,synthetic=SYNTHETIC)
print(json.dumps(dec,indent=2,default=str))

{
  "verdict": "PERP_STATISTICAL_ONLY",
  "status": "HISTORICAL_EXPLORATORY",
  "gates": [
    {
      "asset": "BTC",
      "gate": "stat_threeclass_logloss",
      "delta": -0.00036855456698833817,
      "upper95": -0.00011066137162327751,
      "pass": true
    },
    {
      "asset": "BTC",
      "gate": "stat_binary_logloss",
      "delta": -0.0003652875062578722,
      "upper95": -0.00013496811719277237,
      "pass": true
    },
    {
      "asset": "BTC",
      "gate": "econ_taker_top_decile",
      "net_bp_per_day": -208.3228153551216,
      "lower95": -223.23584584645872,
      "pass": false
    },
    {
      "asset": "ETH",
      "gate": "stat_threeclass_logloss",
      "delta": -0.0007209542944801757,
      "upper95": -0.0003948589490903561,
      "pass": true
    },
    {
      "asset": "ETH",
      "gate": "stat_binary_logloss",
      "delta": -0.0006605908724051883,
      "upper95": -0.0003856015979561731,
      "pass": true
    },
    {
      "asset": "ETH",
      "gat

In [8]:
aggregate={'03c_quality.csv':pd.DataFrame(quality),'03c_scores.csv':scores,'03c_paired_day_bootstrap.csv':bootstrap,
           '03c_economics.csv':econ,'03c_daily_pnl_test_taker.csv':daily,'03c_edge_thresholds.csv':thresholds}
for name,frame in aggregate.items():frame.to_csv(OUT/name,index=False)
meta={'module':'CS03c','registration':pt.registration(),'source':data_description,'synthetic':SYNTHETIC,
      'historical_test_reused':True,'panels':[f'{a}_{t}' for a,t in prepared],'excluded':excluded,'split':splits}
(OUT/'03c_manifest.json').write_text(json.dumps(meta,indent=2,default=str),encoding='utf-8')
(OUT/'03c_decision.json').write_text(json.dumps(dec,indent=2,default=str),encoding='utf-8')
print('Saved',len(aggregate)+2,'outputs to',OUT)

Saved 8 outputs to D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03c
